# Build the APOGEE DR17 × Gaia DR3 XP training set

This notebook constructs the labeled metadata table, applies explicit APOGEE quality cuts, resolves Gaia DR3 source IDs, checks XP-continuous availability, and optionally downloads XP coefficients in resumable batches.

**Important:** every sample count is printed from the data actually loaded. No counts are hard-coded.

> **PS6 repository version.** Interrupted downloads, repeated smoke tests, temporary cleanup cells, and path-debugging cells were removed. Large generated products remain local and are documented in README.md.

# Section 1: imports and configuration

### 1.1 Import libraries and configure project paths

Loads the numerical, table, FITS, and Gaia-query libraries; defines reusable data paths, thresholds, batch settings, and the default XP batch directory.

In [ ]:
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table, vstack
from astroquery.gaia import Gaia

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
BATCH_DIR = DATA_DIR / "apogee_xp_batches_stratified_core_50k"
DATA_DIR.mkdir(exist_ok=True)
BATCH_DIR.mkdir(exist_ok=True)

ALLSTAR_PATH = DATA_DIR / "allStarLite-dr17-synspec_rev1.fits"
ALLSTAR_URL = "https://data.sdss.org/sas/dr17/apogee/spectro/aspcap/dr17/synspec_rev1/allStarLite-dr17-synspec_rev1.fits"

SNR_MIN = 70.0
TEFF_MIN, TEFF_MAX = 3500.0, 7000.0
LOGG_MIN, LOGG_MAX = 0.0, 5.0
BATCH_SIZE = 50
DOWNLOAD_XP = False  # Change to True only after the metadata checks pass.

print("Project root:", PROJECT_ROOT)
print("allStarLite path:", ALLSTAR_PATH)


## Section 2: obtain and inspect allStarLite

The notebook uses the DR17 `synspec_rev1` allStarLite catalog. The download is explicit and resumable at the file level: an existing nonempty file is not replaced.

### 2.1 Download or reuse the APOGEE DR17 catalogue

Downloads `allStarLite` only when the local FITS file is absent, then opens the catalogue for inspection.

In [ ]:
# Download only when the file is absent.
if not ALLSTAR_PATH.exists() or ALLSTAR_PATH.stat().st_size == 0:
    from urllib.request import urlretrieve
    print("Downloading", ALLSTAR_URL)
    urlretrieve(ALLSTAR_URL, ALLSTAR_PATH)
else:
    print("Using existing file:", ALLSTAR_PATH)

with fits.open(ALLSTAR_PATH, memmap=True) as hdul:
    print(hdul.info())
    allstar = Table(hdul[1].data)

print("Loaded rows:", len(allstar))
print("Columns:", len(allstar.colnames))

### 2.2 Validate required APOGEE columns

Checks that the stellar labels, flags, coordinates, and Gaia identifier fields required by the project are present before processing continues.

In [ ]:
# Resolve required columns defensively because FITS readers preserve DR17 names exactly.
REQUIRED = [
    "APOGEE_ID", "RA", "DEC", "SNR", "ASPCAPFLAG", "STARFLAG",
    "TEFF", "LOGG", "FE_H", "N_FE", "O_FE", "AL_FE"
]
missing = [c for c in REQUIRED if c not in allstar.colnames]
if missing:
    raise KeyError(f"Missing required DR17 columns: {missing}")

GAIA_ID_CANDIDATES = ["GAIAEDR3_SOURCE_ID", "GAIA_SOURCE_ID", "GAIA_DR3_SOURCE_ID"]
gaia_id_col = next((c for c in GAIA_ID_CANDIDATES if c in allstar.colnames), None)
print("Gaia ID column found:", gaia_id_col)
print("Target columns:", ["TEFF", "LOGG", "FE_H", "N_FE", "O_FE", "AL_FE"])

## Section 3: define labels and flags

The fifth target is **[N/O]**, computed as `[N/Fe] - [O/Fe]`. APOGEE DR17 supplies calibrated `N_FE` and `O_FE`; there is no need to treat `N_M` as N/O. The notebook retains `AL_FE` directly.

The baseline quality policy rejects `STAR_BAD`, requires
`SNR >= 70`, requires all five target labels to be finite,
and restricts temperature and surface gravity to the selected
ranges. After Gaia DR3 verification and the XP-availability
check, the final quality filter removes sources carrying
`TEFF_WARN`, `LOGG_WARN`, `M_H_WARN`, or `N_M_WARN`.
`VMICRO_WARN` is recorded but is not used as an exclusion
because microturbulence is not a prediction target.

### 3.1 Build the working label table

Converts selected FITS columns into a pandas table, normalizes column types, and derives the `[N/O]` target from `N_FE - O_FE`.

In [ ]:
# Convert selected FITS columns to a pandas DataFrame.
cols = REQUIRED.copy()
if gaia_id_col:
    cols.append(gaia_id_col)
for optional in ["PARAMFLAG", "ELEMFLAG"]:
    if optional in allstar.colnames:
        cols.append(optional)

df = allstar[cols].to_pandas()
df["N_O"] = pd.to_numeric(df["N_FE"], errors="coerce") - pd.to_numeric(df["O_FE"], errors="coerce")
TARGETS = ["TEFF", "LOGG", "FE_H", "N_O", "AL_FE"]

ASPCAP_STAR_BAD_BIT = 23
BAD_MEASUREMENT_MASK = sum(1 << bit for bit in range(8))

def bit_is_set(values, bit):
    values = pd.to_numeric(values, errors="coerce").fillna(0).astype("int64")
    return (values & (1 << bit)) != 0

print("Rows before cuts:", len(df))
print(df[TARGETS].describe().T)

## Section 4: Apply APOGEE quality and label cuts

### 4.1 Apply sequential quality cuts and record cutflow

Applies `STAR_BAD`, signal-to-noise, finite-label, temperature, gravity, and Gaia-ID requirements while recording rows before and after each cut.

In [ ]:
# Section 4: sequential cuts with counts produced from this run.
cut_log = []
current = df.copy()

def apply_cut(name, mask):
    global current
    before = len(current)
    mask = pd.Series(mask, index=current.index).fillna(False)
    current = current.loc[mask].copy()
    cut_log.append({"cut": name, "before": before, "after": len(current), "removed": before-len(current)})
    print(f"{name}: {before:,} -> {len(current):,} (removed {before-len(current):,})")

apply_cut("ASPCAP result is not STAR_BAD", ~bit_is_set(current["ASPCAPFLAG"], ASPCAP_STAR_BAD_BIT))
apply_cut(f"SNR >= {SNR_MIN:g}", pd.to_numeric(current["SNR"], errors="coerce") >= SNR_MIN)
apply_cut("all five targets finite", np.isfinite(current[TARGETS].apply(pd.to_numeric, errors="coerce")).all(axis=1))
apply_cut(f"{TEFF_MIN:g} <= TEFF <= {TEFF_MAX:g}", current["TEFF"].between(TEFF_MIN, TEFF_MAX))
apply_cut(f"{LOGG_MIN:g} <= LOGG <= {LOGG_MAX:g}", current["LOGG"].between(LOGG_MIN, LOGG_MAX))

cut_log_df = pd.DataFrame(cut_log)
cut_log_df.to_csv(DATA_DIR / "apogee_training_cutflow.csv", index=False)
display(cut_log_df)

### Individual abundance/parameter flags

DR17 stores parameter and element flags as arrays in some allStar products. Because their array ordering must match the file's documented parameter/element ordering, this notebook reports their presence rather than guessing indices. The conservative `STAR_BAD` filter above is always applied. Before the final science sample is frozen, inspect the file header/data model and add per-label BAD-bit filtering if the array ordering is present and verified.

### 4.2 Inspect available warning and parameter flags

Reports the flag fields available in this `allStarLite` product and summarizes warning-bit counts used to justify the final quality policy.

In [ ]:
print("PARAMFLAG present:", "PARAMFLAG" in allstar.colnames)
print("ELEMFLAG present:", "ELEMFLAG" in allstar.colnames)
print("ASPCAP STAR_WARN count in retained sample:", int(bit_is_set(current["ASPCAPFLAG"], 7).sum()))
print("ASPCAP STAR_BAD count in retained sample:", int(bit_is_set(current["ASPCAPFLAG"], 23).sum()))

def summarize_aspcap_warnings(frame, sample_name):
    counts = pd.Series({
        "TEFF_WARN": bit_is_set(frame["ASPCAPFLAG"], 0).sum(),
        "LOGG_WARN": bit_is_set(frame["ASPCAPFLAG"], 1).sum(),
        "VMICRO_WARN": bit_is_set(frame["ASPCAPFLAG"], 2).sum(),
        "M_H_WARN": bit_is_set(frame["ASPCAPFLAG"], 3).sum(),
        "N_M_WARN": bit_is_set(frame["ASPCAPFLAG"], 6).sum(),
        "STAR_WARN": bit_is_set(frame["ASPCAPFLAG"], 7).sum(),
        "STAR_BAD": bit_is_set(frame["ASPCAPFLAG"], 23).sum(),
    })

    summary = pd.DataFrame({
        "count": counts,
        "percent": 100 * counts / len(frame)
    })

    print(f"{sample_name}: {len(frame):,} rows")
    return summary

## Section 5: Gaia DR3 source IDs

Preferred strategy: use the catalog's Gaia EDR3 source ID directly, because Gaia EDR3 and Gaia DR3 use the same source identifiers. Then verify every ID against `gaiadr3.gaia_source` and require `has_xp_continuous = TRUE`. If the catalog lacks a Gaia ID column, stop rather than silently performing an ambiguous positional match; a proper-motion-aware Gaia Archive crossmatch should then be added as a separate audited step.

### 5.1 Standardize and validate Gaia source identifiers

Converts APOGEE-provided Gaia identifiers to integer `source_id` values, removes invalid identifiers, and verifies uniqueness.

In [ ]:
if gaia_id_col is None:
    raise RuntimeError(
        "No Gaia source-ID column was found. Add a proper-motion-aware positional crossmatch; "
        "do not continue with nearest-neighbour matching silently."
    )

current["source_id"] = pd.to_numeric(current[gaia_id_col], errors="coerce").astype("Int64")
apply_cut("valid positive Gaia source_id", current["source_id"].notna() & (current["source_id"] > 0))
apply_cut("one row per Gaia source_id", ~current["source_id"].duplicated(keep="first"))

upload = Table.from_pandas(current[["source_id"]].astype({"source_id":"int64"}))
upload_path = DATA_DIR / "apogee_gaia_ids.vot"
upload.write(upload_path, format="votable", overwrite=True)
print("Upload table rows:", len(upload))

# Section 6: verify IDs and XP availability in Gaia DR3

### 6.1 Query Gaia DR3 in restart-safe ID batches

Checks submitted identifiers against `gaiadr3.gaia_source`, records XP-continuous availability, and saves completed query batches for reuse.

In [ ]:
# Section 6: verify Gaia IDs and XP availability in chunks.

from astropy.table import Table
import pandas as pd
import time

GAIA_QUERY_BATCH_SIZE = 40_000
GAIA_QUERY_PAUSE = 2
GAIA_QUERY_RETRIES = 3

gaia_query_dir = DATA_DIR / "apogee_gaia_query_batches"
gaia_query_dir.mkdir(exist_ok=True)

Gaia.ROW_LIMIT = -1


def query_gaia_batch(batch_df, batch_number):
    """
    Upload one source-ID batch and join it to gaiadr3.gaia_source.
    Existing completed CSV files are reused when the notebook resumes.
    """
    batch_output = (
        gaia_query_dir /
        f"gaia_xp_check_{batch_number:04d}.csv"
    )

    # Resume from an existing successful batch.
    if batch_output.exists() and batch_output.stat().st_size > 0:
        saved = pd.read_csv(
            batch_output,
            dtype={"source_id": "int64"}
        )

        if len(saved) > len(batch_df):
            raise RuntimeError(
                f"{batch_output.name} contains more rows than its input batch."
            )

        print(
            f"Batch {batch_number}: reused "
            f"{len(saved):,} existing rows"
        )
        return saved

    batch_upload = Table.from_pandas(
        batch_df[["source_id"]].astype({"source_id": "int64"})
    )

    batch_upload_path = (
        DATA_DIR /
        f"apogee_gaia_ids_batch_{batch_number:04d}.vot"
    )

    batch_upload.write(
        batch_upload_path,
        format="votable",
        overwrite=True
    )

    adql = """
    SELECT
        u.source_id,
        g.has_xp_continuous
    FROM tap_upload.apogee_ids AS u
    JOIN gaiadr3.gaia_source AS g
        ON g.source_id = u.source_id
    """

    last_error = None

    for attempt in range(1, GAIA_QUERY_RETRIES + 1):
        try:
            print(
                f"Batch {batch_number}, attempt {attempt}: "
                f"submitting {len(batch_df):,} IDs"
            )

            job = Gaia.launch_job_async(
                adql,
                upload_resource=str(batch_upload_path),
                upload_table_name="apogee_ids",
                dump_to_file=False,
            )

            result = job.get_results().to_pandas()

            result["source_id"] = pd.to_numeric(
                result["source_id"],
                errors="raise"
            ).astype("int64")

            # A one-to-one source-ID join cannot return more rows
            # than were submitted.
            if len(result) > len(batch_df):
                raise RuntimeError(
                    f"Batch {batch_number} returned {len(result):,} rows "
                    f"for only {len(batch_df):,} submitted IDs."
                )

            if len(result) == 50_000:
                raise RuntimeError(
                    f"Batch {batch_number} reached the 50,000-row cap."
                )

            if result["source_id"].duplicated().any():
                raise RuntimeError(
                    f"Batch {batch_number} returned duplicate source IDs."
                )

            result.to_csv(batch_output, index=False)

            print(
                f"Batch {batch_number}: "
                f"{len(result):,} IDs verified, "
                f"{int(result['has_xp_continuous'].fillna(False).sum()):,} "
                f"with XP"
            )

            return result

        except Exception as exc:
            last_error = exc
            print(
                f"Batch {batch_number}, attempt {attempt} failed: {exc}"
            )

            if attempt < GAIA_QUERY_RETRIES:
                time.sleep(30 * attempt)

    raise RuntimeError(
        f"Batch {batch_number} failed after "
        f"{GAIA_QUERY_RETRIES} attempts."
    ) from last_error

# Ensure the submitted source IDs are unique.
query_ids = (
    current[["source_id"]]
    .dropna()
    .copy()
)

query_ids["source_id"] = query_ids["source_id"].astype("int64")
query_ids = query_ids.drop_duplicates("source_id").reset_index(drop=True)

print("Total unique IDs to submit:", len(query_ids))

gaia_batch_results = []

for start in range(0, len(query_ids), GAIA_QUERY_BATCH_SIZE):
    stop = min(start + GAIA_QUERY_BATCH_SIZE, len(query_ids))
    batch_number = start // GAIA_QUERY_BATCH_SIZE

    batch_df = query_ids.iloc[start:stop].copy()

    result = query_gaia_batch(
        batch_df=batch_df,
        batch_number=batch_number
    )

    gaia_batch_results.append(result)

    if stop < len(query_ids):
        time.sleep(GAIA_QUERY_PAUSE)

gaia_check = pd.concat(
    gaia_batch_results,
    ignore_index=True
)

gaia_check["source_id"] = pd.to_numeric(
    gaia_check["source_id"],
    errors="raise"
).astype("int64")

print("Combined Gaia rows:", len(gaia_check))
print(
    "Unique combined Gaia IDs:",
    gaia_check["source_id"].nunique()
)
print(
    "Duplicate combined Gaia IDs:",
    gaia_check["source_id"].duplicated().sum()
)
print(
    "Combined IDs with XP continuous:",
    int(
        gaia_check["has_xp_continuous"]
        .fillna(False)
        .sum()
    )
)

if gaia_check["source_id"].duplicated().any():
    duplicate_ids = gaia_check.loc[
        gaia_check["source_id"].duplicated(keep=False),
        "source_id"
    ]

    raise RuntimeError(
        f"The combined Gaia result contains "
        f"{duplicate_ids.nunique():,} duplicated source IDs."
    )

unexpected_ids = set(gaia_check["source_id"]) - set(query_ids["source_id"])

if unexpected_ids:
    raise RuntimeError(
        f"Gaia returned {len(unexpected_ids):,} IDs "
        f"that were not submitted."
    )

submitted_ids = set(query_ids["source_id"])
verified_ids = set(gaia_check["source_id"])

unverified_ids = sorted(submitted_ids - verified_ids)

print("Submitted IDs:", len(submitted_ids))
print("Verified Gaia DR3 IDs:", len(verified_ids))
print("IDs not returned:", len(unverified_ids))

pd.DataFrame({
    "source_id": unverified_ids
}).to_csv(
    DATA_DIR / "apogee_gaia_unverified_ids.csv",
    index=False
)

training_meta = current.merge(
    gaia_check,
    on="source_id",
    how="inner",
    validate="one_to_one"
)

training_meta = training_meta.loc[
    training_meta["has_xp_continuous"].fillna(False)
].copy()

print("Rows after APOGEE and source-ID cuts:", len(current))
print("Unique IDs submitted:", len(query_ids))
print("IDs verified in Gaia DR3:", len(gaia_check))
print("IDs not returned by Gaia DR3:", len(unverified_ids))
print("Rows with XP continuous:", len(training_meta))
print(
    "Unique training source IDs:",
    training_meta["source_id"].nunique()
)
print(
    "Duplicate training source IDs:",
    training_meta["source_id"].duplicated().sum()
)

keep_columns = [
    "source_id",
    "APOGEE_ID",
    "RA",
    "DEC",
    "SNR",
    "TEFF",
    "LOGG",
    "FE_H",
    "N_FE",
    "O_FE",
    "N_O",
    "AL_FE",
    "ASPCAPFLAG",
    "STARFLAG",
]

training_meta[keep_columns].to_csv(
    DATA_DIR /
    "apogee_dr17_gaia_dr3_xp_training_metadata.csv",
    index=False
)

print("Saved complete training metadata:", len(training_meta))

warning_counts = pd.Series({
    "TEFF_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 0
    ).sum(),

    "LOGG_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 1
    ).sum(),

    "VMICRO_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 2
    ).sum(),

    "M_H_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 3
    ).sum(),

    "N_M_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 6
    ).sum(),

    "STAR_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 7
    ).sum(),

    "STAR_BAD": bit_is_set(
        training_meta["ASPCAPFLAG"], 23
    ).sum(),
})

warning_summary = pd.DataFrame({
    "count": warning_counts,
    "percent": (
        100 * warning_counts / len(training_meta)
    )
})

display(warning_summary)

### 6.2 Summarize Gaia verification and define target-warning mask

Reports returned and unreturned Gaia identifiers and combines the target-related ASPCAP warning bits used for the final label-quality filter.

In [ ]:
print("IDs verified in Gaia DR3:", len(gaia_check))
print("IDs not returned:", len(unverified_ids))

target_warning_mask = (
    bit_is_set(training_meta["ASPCAPFLAG"], 0) |  # TEFF_WARN
    bit_is_set(training_meta["ASPCAPFLAG"], 1) |  # LOGG_WARN
    bit_is_set(training_meta["ASPCAPFLAG"], 3) |  # M_H_WARN
    bit_is_set(training_meta["ASPCAPFLAG"], 6)    # N_M_WARN
)

print("Current XP sample:", len(training_meta))
print("Rows with a target-related warning:", target_warning_mask.sum())
print("Rows remaining:", (~target_warning_mask).sum())
print(
    "Percent retained:",
    100 * (~target_warning_mask).mean()
)

policies = {
    "Target warnings only": target_warning_mask,

    "Any STAR_WARN": bit_is_set(
        training_meta["ASPCAPFLAG"], 7
    ),

    "Target warnings or STAR_WARN": (
        target_warning_mask |
        bit_is_set(training_meta["ASPCAPFLAG"], 7)
    ),

    "Target warnings or VMICRO_WARN": (
        target_warning_mask |
        bit_is_set(training_meta["ASPCAPFLAG"], 2)
    ),
}

for name, mask in policies.items():
    print(
        f"{name}: remove {mask.sum():,}, "
        f"retain {(~mask).sum():,} "
        f"({100 * (~mask).mean():.2f}%)"
    )

### 6.3 Create the final quality-filtered metadata table

Removes rows carrying target-related warnings and verifies the final source count and identifier uniqueness.

In [ ]:
training_meta_quality = (
    training_meta.loc[~target_warning_mask]
    .copy()
    .reset_index(drop=True)
)

print("Before target-warning filter:", len(training_meta))
print("After target-warning filter:", len(training_meta_quality))
print(
    "Duplicate source IDs:",
    training_meta_quality["source_id"].duplicated().sum()
)

training_meta_quality.to_csv(
    DATA_DIR /
    "apogee_dr17_gaia_dr3_xp_training_metadata_quality.csv",
    index=False
)

final_cutflow = pd.DataFrame([
    {
        "cut": "Valid unique Gaia source ID",
        "before": 495464,
        "after": 448728,
        "removed": 46736,
    },
    {
        "cut": "Verified in Gaia DR3",
        "before": 448728,
        "after": len(gaia_check),
        "removed": 448728 - len(gaia_check),
    },
    {
        "cut": "Gaia XP continuous available",
        "before": len(gaia_check),
        "after": len(training_meta),
        "removed": len(gaia_check) - len(training_meta),
    },
    {
        "cut": "No target-related ASPCAP warnings",
        "before": len(training_meta),
        "after": len(training_meta_quality),
        "removed": len(training_meta) - len(training_meta_quality),
    },
])

complete_cutflow = pd.concat(
    [
        pd.read_csv(DATA_DIR / "apogee_training_cutflow.csv"),
        final_cutflow,
    ],
    ignore_index=True,
)

complete_cutflow.to_csv(
    DATA_DIR / "apogee_training_cutflow_complete.csv",
    index=False
)

display(complete_cutflow)

## Section 7: optional resumable XP download

Each completed batch is stored separately. Rerunning skips existing nonempty FITS batches. HTTP failures are recorded and do not erase completed work.

### 7.1 Define restart-safe XP download and validation helpers

Defines batch retrieval, retry, FITS validation, progress logging, and failure-recording functions without starting a new download automatically.

In [ ]:
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.table import Table


XP_PRODUCT_KEY = "XP_CONTINUOUS_DATAMODEL_GAIA.fits"


def validate_xp_batch(path, requested_ids=None):
    """
    Verify that a saved Gaia XP FITS batch is readable and contains
    the required source ID and coefficient columns.
    """
    path = Path(path)
    table = Table.read(path)

    required_columns = {
        "source_id",
        "bp_coefficients",
        "rp_coefficients",
    }

    missing_columns = required_columns - set(table.colnames)

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    source_ids = np.asarray(
        table["source_id"],
        dtype=np.int64,
    )

    if len(source_ids) != len(np.unique(source_ids)):
        raise ValueError(
            f"{path.name} contains duplicate source IDs"
        )

    bp_lengths = np.asarray([
        len(np.asarray(values))
        for values in table["bp_coefficients"]
    ])

    rp_lengths = np.asarray([
        len(np.asarray(values))
        for values in table["rp_coefficients"]
    ])

    invalid_lengths = (
        (bp_lengths != 55)
        | (rp_lengths != 55)
    )

    if invalid_lengths.any():
        raise ValueError(
            f"{path.name} contains "
            f"{int(invalid_lengths.sum())} rows without "
            f"55 BP and 55 RP coefficients"
        )

    if requested_ids is not None:
        requested_set = {
            int(value)
            for value in requested_ids
        }

        returned_set = {
            int(value)
            for value in source_ids
        }

        unexpected_ids = returned_set - requested_set

        if unexpected_ids:
            raise ValueError(
                f"{path.name} contains "
                f"{len(unexpected_ids)} unrequested source IDs"
            )

    return table


def save_gaia_product(product, output_path):
    """
    Save a Gaia DataLink product to a FITS file.

    Gaia.load_data() may return an Astropy Table directly or, in some
    astroquery versions, a path to a downloaded file.
    """
    output_path = Path(output_path)

    if isinstance(product, Table):
        print(
            "Returned Gaia product type:",
            type(product),
        )

        product.write(
            output_path,
            format="fits",
            overwrite=True,
        )

        return

    if isinstance(product, (str, Path)):
        source_path = Path(product)

        print(
            "Returned Gaia product path:",
            source_path,
        )

        if not source_path.exists():
            raise FileNotFoundError(
                f"Returned file does not exist: {source_path}"
            )

        shutil.copy2(
            source_path,
            output_path,
        )

        return

    raise TypeError(
        "Unsupported Gaia product type: "
        f"{type(product)}"
    )


def download_xp_batches(
    source_ids,
    batch_size=BATCH_SIZE,
    max_retries=3,
):
    """
    Download Gaia DR3 XP continuous coefficients in restart-safe
    FITS batches.
    """
    source_ids = (
        pd.Series(source_ids, dtype="int64")
        .drop_duplicates()
        .tolist()
    )

    failures = []
    progress = []

    total_batches = (
        len(source_ids) + batch_size - 1
    ) // batch_size

    print("Target source IDs:", f"{len(source_ids):,}")
    print("Batch size:", f"{batch_size:,}")
    print("Total batches:", f"{total_batches:,}")

    for start in range(
        0,
        len(source_ids),
        batch_size,
    ):
        batch_num = start // batch_size

        ids = source_ids[
            start:start + batch_size
        ]

        out = (
            BATCH_DIR
            / f"apogee_xp_{batch_num:05d}.fits"
        )

        partial_path = out.with_suffix(
            ".partial.fits"
        )

        # Remove a partial file left by an interrupted request.
        if partial_path.exists():
            partial_path.unlink()

            print(
                "Removed incomplete file:",
                partial_path.name,
            )

        # Reuse an existing batch only after validation.
        if out.exists() and out.stat().st_size > 0:
            try:
                existing = validate_xp_batch(
                    out,
                    requested_ids=ids,
                )

                existing_ids = {
                    int(value)
                    for value in np.asarray(
                        existing["source_id"],
                        dtype=np.int64,
                    )
                }

                existing_missing = (
                    set(ids) - existing_ids
                )

                print(
                    f"Batch {batch_num + 1:,}/"
                    f"{total_batches:,}: "
                    f"reused {len(existing):,} rows; "
                    f"{len(existing_missing):,} missing"
                )

                progress.append({
                    "batch": batch_num,
                    "requested": len(ids),
                    "returned": len(existing),
                    "missing": len(existing_missing),
                    "status": "reused",
                    "file": out.name,
                })

                for source_id in sorted(
                    existing_missing
                ):
                    failures.append({
                        "batch": batch_num,
                        "source_id": source_id,
                        "reason": (
                            "not present in existing batch"
                        ),
                    })

                continue

            except Exception as error:
                invalid_path = (
                    BATCH_DIR
                    / (
                        f"apogee_xp_"
                        f"{batch_num:05d}"
                        f".invalid.fits"
                    )
                )

                if invalid_path.exists():
                    invalid_path.unlink()

                out.replace(invalid_path)

                print(
                    f"Existing batch {out.name} failed "
                    f"validation and was renamed to "
                    f"{invalid_path.name}"
                )

                print(
                    "Validation error:",
                    error,
                )

        completed = False
        last_error = None

        for attempt in range(
            1,
            max_retries + 1,
        ):
            try:
                print(
                    f"Batch {batch_num + 1:,}/"
                    f"{total_batches:,}, "
                    f"attempt {attempt}: requesting "
                    f"{len(ids):,} sources"
                )

                result = Gaia.load_data(
                    ids=ids,
                    data_release="Gaia DR3",
                    retrieval_type=(
                        "XP_CONTINUOUS"
                    ),
                    data_structure=(
                        "DATAMODEL_GAIA"
                    ),
                    format="fits",
                )

                if XP_PRODUCT_KEY not in result:
                    raise KeyError(
                        f"Expected {XP_PRODUCT_KEY}, "
                        f"but Gaia returned "
                        f"{list(result.keys())}"
                    )

                returned_products = result[
                    XP_PRODUCT_KEY
                ]

                if not returned_products:
                    raise RuntimeError(
                        "Gaia returned an empty "
                        "product list"
                    )

                product = returned_products[0]

                save_gaia_product(
                    product,
                    partial_path,
                )

                # Validate before marking the batch complete.
                table = validate_xp_batch(
                    partial_path,
                    requested_ids=ids,
                )

                # Rename only after successful validation.
                partial_path.replace(out)

                returned_ids = {
                    int(value)
                    for value in np.asarray(
                        table["source_id"],
                        dtype=np.int64,
                    )
                }

                missing_ids = sorted(
                    set(ids) - returned_ids
                )

                print(
                    f"Saved {out.name}: "
                    f"{len(table):,} rows; "
                    f"{len(missing_ids):,} missing"
                )

                progress.append({
                    "batch": batch_num,
                    "requested": len(ids),
                    "returned": len(table),
                    "missing": len(missing_ids),
                    "status": "downloaded",
                    "file": out.name,
                })

                for source_id in missing_ids:
                    failures.append({
                        "batch": batch_num,
                        "source_id": source_id,
                        "reason": (
                            "not returned by Gaia"
                        ),
                    })

                completed = True
                break

            except Exception as error:
                last_error = error

                if partial_path.exists():
                    partial_path.unlink()

                print(
                    f"Batch {batch_num}, "
                    f"attempt {attempt} failed:"
                )

                print(error)

                if attempt < max_retries:
                    wait_seconds = (
                        30 * attempt
                    )

                    print(
                        f"Waiting {wait_seconds} "
                        f"seconds before retrying"
                    )

                    time.sleep(wait_seconds)

        if not completed:
            for source_id in ids:
                failures.append({
                    "batch": batch_num,
                    "source_id": source_id,
                    "reason": repr(last_error),
                })

            progress.append({
                "batch": batch_num,
                "requested": len(ids),
                "returned": 0,
                "missing": len(ids),
                "status": "failed",
                "file": out.name,
            })

        # Save progress after every batch.
        progress_columns = [
            "batch",
            "requested",
            "returned",
            "missing",
            "status",
            "file",
        ]

        failure_columns = [
            "batch",
            "source_id",
            "reason",
        ]

        pd.DataFrame(
            progress,
            columns=progress_columns,
        ).to_csv(
            DATA_DIR
            / "apogee_xp_batch_progress.csv",
            index=False,
        )

        pd.DataFrame(
            failures,
            columns=failure_columns,
        ).to_csv(
            DATA_DIR
            / "apogee_xp_download_failures.csv",
            index=False,
        )

        time.sleep(2)

    print("Download loop finished.")

    print(
        "Failed or missing source IDs:",
        f"{len(failures):,}",
    )

    return (
        pd.DataFrame(
            progress,
            columns=[
                "batch",
                "requested",
                "returned",
                "missing",
                "status",
                "file",
            ],
        ),
        pd.DataFrame(
            failures,
            columns=[
                "batch",
                "source_id",
                "reason",
            ],
        ),
    )


if DOWNLOAD_XP:
    progress_df, failures_df = (
        download_xp_batches(
            training_meta_quality[
                "source_id"
            ],
            batch_size=BATCH_SIZE,
            max_retries=3,
        )
    )

    print(
        "Failed or missing IDs:",
        len(failures_df),
    )

else:
    print(
        "XP download disabled. "
        "Set DOWNLOAD_XP = True after "
        "the saved-batch test succeeds."
    )

### 7.2 Inventory existing downloaded XP batches

Reads existing batch files, collects downloaded source identifiers, and identifies unreadable or duplicate products before selecting additional sources.

In [ ]:
from astropy.table import Table
import numpy as np
import pandas as pd


existing_batch_files = sorted(
    BATCH_DIR.glob("apogee_xp_*.fits")
)

downloaded_id_parts = []
unreadable_batch_files = []

for path in existing_batch_files:
    try:
        table = Table.read(path)

        if "source_id" not in table.colnames:
            raise KeyError(
                f"{path.name} does not contain source_id"
            )

        downloaded_id_parts.append(
            np.asarray(
                table["source_id"],
                dtype=np.int64,
            )
        )

    except Exception as error:
        unreadable_batch_files.append({
            "file": path.name,
            "error": repr(error),
        })

        print(
            "Could not read:",
            path.name,
            error,
        )


if downloaded_id_parts:
    downloaded_ids = np.unique(
        np.concatenate(downloaded_id_parts)
    )

else:
    downloaded_ids = np.array(
        [],
        dtype=np.int64,
    )


print(
    "Existing batch files:",
    len(existing_batch_files),
)

print(
    "Readable batch files:",
    len(existing_batch_files)
    - len(unreadable_batch_files),
)

print(
    "Unreadable batch files:",
    len(unreadable_batch_files),
)

print(
    "Unique sources downloaded:",
    len(downloaded_ids),
)

pd.DataFrame(
    unreadable_batch_files,
    columns=["file", "error"],
).to_csv(
    DATA_DIR / "apogee_xp_unreadable_batches.csv",
    index=False,
)

### 7.3 Construct the first stratified 50,000-source selection

Stratifies the quality-filtered metadata across metallicity and gravity bins so the download subset covers more of the label space.

In [ ]:
from sklearn.model_selection import train_test_split

selection_meta = (
    training_meta_quality
    .copy()
    .reset_index(drop=True)
)

selection_meta["feh_bin"] = pd.qcut(
    selection_meta["FE_H"],
    q=20,
    duplicates="drop",
)

selection_meta["logg_bin"] = pd.qcut(
    selection_meta["LOGG"],
    q=10,
    duplicates="drop",
)

selection_meta["download_stratum"] = (
    selection_meta["feh_bin"]
    .astype(str)
    + "__"
    + selection_meta["logg_bin"]
    .astype(str)
)

stratified_core, _ = train_test_split(
    selection_meta,
    train_size=50_000,
    random_state=42,
    shuffle=True,
    stratify=selection_meta["download_stratum"],
)

stratified_core = (
    stratified_core
    .sample(
        frac=1.0,
        random_state=42,
    )
    .reset_index(drop=True)
)

print("Stratified core rows:", len(stratified_core))
print(
    "Unique stratified IDs:",
    stratified_core["source_id"].nunique(),
)

### 7.4 Remove already downloaded identifiers from the core selection

Compares the stratified core IDs against existing XP products and builds the remaining source list required for restart-safe retrieval.

In [ ]:
downloaded_id_set = {
    int(value)
    for value in downloaded_ids
}

stratified_ids = (
    stratified_core["source_id"]
    .astype("int64")
)

stratified_remaining_ids = (
    stratified_ids.loc[
        ~stratified_ids.isin(
            downloaded_id_set
        )
    ]
    .reset_index(drop=True)
)

already_in_stratified_core = (
    len(stratified_ids)
    - len(stratified_remaining_ids)
)

print(
    "Stratified target:",
    len(stratified_ids),
)

print(
    "Already downloaded from target:",
    already_in_stratified_core,
)

print(
    "Still needed for stratified target:",
    len(stratified_remaining_ids),
)

stratified_core[[
    "source_id",
    "TEFF",
    "LOGG",
    "FE_H",
    "N_O",
    "AL_FE",
]].to_csv(
    DATA_DIR /
    "apogee_training_stratified_core_50k.csv",
    index=False,
)

stratified_remaining_ids.to_frame(
    name="source_id"
).to_csv(
    DATA_DIR /
    "apogee_training_stratified_remaining_ids.csv",
    index=False,
)

### 7.5 Construct the 50,000-source extension selection

Builds a second stratified subset from sources not used by the core selection, allowing the project to expand toward a 100,000-source target.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd


selection_meta = (
    training_meta_quality
    .copy()
    .reset_index(drop=True)
)

selection_meta["feh_bin"] = pd.qcut(
    selection_meta["FE_H"],
    q=20,
    duplicates="drop",
)

selection_meta["logg_bin"] = pd.qcut(
    selection_meta["LOGG"],
    q=10,
    duplicates="drop",
)

selection_meta["download_stratum"] = (
    selection_meta["feh_bin"].astype(str)
    + "__"
    + selection_meta["logg_bin"].astype(str)
)

stratified_100k, _ = train_test_split(
    selection_meta,
    train_size=100_000,
    random_state=42,
    shuffle=True,
    stratify=selection_meta["download_stratum"],
)

stratified_100k = (
    stratified_100k
    .sample(
        frac=1.0,
        random_state=42,
    )
    .reset_index(drop=True)
)

stratified_50k = (
    stratified_100k
    .iloc[:50_000]
    .copy()
)

stratified_extension_50k = (
    stratified_100k
    .iloc[50_000:100_000]
    .copy()
)

print("Full target:", len(stratified_100k))
print("Initial core:", len(stratified_50k))
print(
    "Later extension:",
    len(stratified_extension_50k),
)

### 7.6 Verify that core and extension selections do not overlap

Checks uniqueness within both selections and confirms that no source identifier appears in both groups.

In [ ]:
core_ids = set(
    stratified_50k["source_id"]
)

extension_ids = set(
    stratified_extension_50k["source_id"]
)

print("Core unique IDs:", len(core_ids))
print("Extension unique IDs:", len(extension_ids))
print("Overlap:", len(core_ids & extension_ids))
print("Union:", len(core_ids | extension_ids))

### 7.7 Save reproducible stratified selection tables

Sorts and writes the core, extension, and combined 100,000-source metadata selections used by subsequent download and audit steps.

In [ ]:
stratified_100k = (
    stratified_100k
    .sort_values(
        [
            "download_stratum",
            "source_id",
        ]
    )
    .reset_index(drop=True)
)

stratified_100k["phase"] = (
    stratified_100k
    .groupby(
        "download_stratum",
        observed=True,
    )
    .cumcount()
    % 2
)

stratified_50k = (
    stratified_100k.loc[
        stratified_100k["phase"] == 0
    ]
    .sample(
        frac=1.0,
        random_state=42,
    )
    .head(50_000)
    .reset_index(drop=True)
)

core_id_set = set(
    stratified_50k["source_id"]
)

stratified_extension_50k = (
    stratified_100k.loc[
        ~stratified_100k["source_id"].isin(
            core_id_set
        )
    ]
    .sample(
        frac=1.0,
        random_state=42,
    )
    .head(50_000)
    .reset_index(drop=True)
)

### 7.8 Validate all available XP coefficient rows

Reads the downloaded FITS files, counts rows, checks coefficient arrays, records unreadable files, and gathers available source identifiers.

In [ ]:
from astropy.table import Table
import numpy as np

total_rows = 0
empty_files = []
bad_files = []
source_ids = []

for file in files:
    try:
        table = Table.read(file)
        total_rows += len(table)

        if len(table) == 0:
            empty_files.append(file.name)

        if "source_id" in table.colnames:
            source_ids.extend(
                np.asarray(table["source_id"])
                .astype(np.int64)
                .tolist()
            )

    except Exception as exc:
        bad_files.append((file.name, str(exc)))

print("FITS files:", len(files))
print("Total rows:", total_rows)
print("Unique source IDs:", len(set(source_ids)))
print(
    "Duplicate source IDs:",
    len(source_ids) - len(set(source_ids))
)
print("Empty files:", len(empty_files))
print("Unreadable files:", len(bad_files))

if bad_files:
    print("\nUnreadable files:")
    for name, error in bad_files[:10]:
        print(name, "->", error)

### 7.9 Identify missing batch indices

Compares the expected 973 batch indices with the files present on disk to quantify the 80 missing batches.

In [ ]:
expected_indices = set(range(973))

downloaded_indices = {
    int(file.stem.split("_")[-1])
    for file in files
}

missing_indices = sorted(
    expected_indices - downloaded_indices
)

print("Missing batch files:", len(missing_indices))
print("Missing indices:", missing_indices)

### 7.10 Recheck row counts and source-ID uniqueness

Performs a final independent inventory of downloaded rows, empty files, unreadable files, and duplicate identifiers.

In [ ]:
from astropy.table import Table
import numpy as np

total_rows = 0
empty_files = []
bad_files = []
source_ids = []

for file in batch_files:
    try:
        table = Table.read(file)
        total_rows += len(table)

        if len(table) == 0:
            empty_files.append(file.name)

        if "source_id" in table.colnames:
            ids = np.asarray(table["source_id"]).astype(np.int64)
            source_ids.extend(ids.tolist())

    except Exception as exc:
        bad_files.append((file.name, str(exc)))

print("FITS files:", len(batch_files))
print("Total rows:", total_rows)
print("Unique source IDs:", len(set(source_ids)))
print("Duplicate source IDs:",
      len(source_ids) - len(set(source_ids)))
print("Empty files:", len(empty_files))
print("Unreadable files:", len(bad_files))

### 7.11 Save the available XP source-ID list

Writes the exact set of successfully downloaded source identifiers so later preprocessing uses only available data.

In [ ]:
import pandas as pd

available_ids = pd.DataFrame({
    "source_id": sorted(set(source_ids))
})

available_ids_path = (
    DATA_DIR / "apogee_xp_available_core_source_ids.csv"
)

available_ids.to_csv(available_ids_path, index=False)

print("Saved:", available_ids_path)
print("Available sources:", len(available_ids))

## Section 8: Save final audit outputs

Run this after the preceding sections. It writes only observed counts and current configuration values.

### 8.1 Save final metadata and audit summaries

Writes the quality-filtered metadata, cutflow, configuration, and observed project counts used by the report and reproducibility workflow.

In [ ]:
quality_path = (
    DATA_DIR /
    "apogee_dr17_gaia_dr3_xp_training_metadata_quality.csv"
)

training_meta_quality.to_csv(
    quality_path,
    index=False
)

print("Saved:", quality_path)
print("Rows:", len(training_meta_quality))

training_meta_quality[["source_id"]].to_csv(
    DATA_DIR / "apogee_training_xp_source_ids.csv",
    index=False
)

complete_cutflow.to_csv(
    DATA_DIR / "apogee_training_cutflow.csv",
    index=False
)

summary = {
    "allstar_rows_loaded": 733901,
    "rows_after_star_bad_cut": 691591,
    "rows_after_snr_cut": 597478,
    "rows_after_finite_target_cut": 495611,
    "rows_after_teff_cut": 495611,
    "rows_after_logg_cut": 495464,
    "valid_unique_gaia_ids": 448728,
    "gaia_dr3_ids_verified": 448728,
    "gaia_dr3_ids_not_returned": 0,
    "rows_with_xp_continuous": 405160,
    "rows_after_target_warning_filter": 334486,
    "duplicate_final_source_ids": 0,
    "snr_min": 70,
    "teff_range": [3500, 7000],
    "logg_range": [0, 5],
    "targets": ["TEFF", "LOGG", "FE_H", "N_O", "AL_FE"],
    "n_o_definition": "N_FE - O_FE",
    "excluded_warning_bits": [
        "TEFF_WARN",
        "LOGG_WARN",
        "M_H_WARN",
        "N_M_WARN"
    ]
}

import json

with open(
    DATA_DIR / "apogee_training_summary.json",
    "w"
) as file:
    json.dump(summary, file, indent=2)